# Application Inference from Packet Flows — Reference Solution

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CrashingGuru/Sandbox_Training/blob/main/notebooks/application-inference-packet-flows-reference/notebook/Application_Inference_Packet_Flows_Reference.ipynb)

Reference solution for the ITU AI/ML Challenge **Application Inference from Packet Flows**. It trains a Random Forest on header-derived flow statistics to classify each flow's mobile application, for both tasks, then writes a `submission.json`.

Click **Open in Colab** to run end-to-end — the data is loaded directly from the public challenge repository, so no download or setup is needed. Test labels are private, so models are scored here on the **validation** split as a proxy.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score
import json

RANDOM_STATE = 42
BASE = (
    "https://raw.githubusercontent.com/FG-AI4H/"
    "application-inference-packet-flows-public/main/dataset"
)
TASKS = ["task1_static", "task2_realtime_prefix5"]
DROP = ["flow_id", "split"]  # identifiers, not features

## 1. Load & preprocess

Features are merged with labels on `flow_id`; the categorical `transport` column is one-hot encoded, and columns are aligned across train/validation/test.

In [ ]:
def load_csv(task, name):
    return pd.read_csv(f"{BASE}/{task}/{name}")

def prepare(task):
    tr = load_csv(task, "train_features.csv").merge(
        load_csv(task, "train_labels.csv"), on="flow_id")
    va = load_csv(task, "validation_features.csv").merge(
        load_csv(task, "validation_labels.csv"), on="flow_id")
    te = load_csv(task, "test_features.csv")

    def feats(df):
        X = df.drop(columns=[c for c in DROP + ["app"] if c in df.columns])
        return pd.get_dummies(X)  # one-hot encodes 'transport'

    X_tr, y_tr = feats(tr), tr["app"]
    X_va, y_va = feats(va), va["app"]
    X_te = feats(te)
    # Align validation/test columns to the training feature space.
    X_va = X_va.reindex(columns=X_tr.columns, fill_value=0)
    X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)
    return X_tr, y_tr, X_va, y_va, X_te, te["flow_id"]

## 2. Train, evaluate and predict — both tasks

In [ ]:
submission = {}
scores = {}
for task in TASKS:
    X_tr, y_tr, X_va, y_va, X_te, test_ids = prepare(task)
    clf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE)
    clf.fit(X_tr, y_tr)

    val_pred = clf.predict(X_va)
    macro_f1 = f1_score(y_va, val_pred, average="macro")
    acc = accuracy_score(y_va, val_pred)
    scores[task] = {"macro_f1": macro_f1, "accuracy": acc}
    print(f"{task}: validation macro-F1 = {macro_f1:.3f}, accuracy = {acc:.3f}")

    submission[task] = dict(zip(test_ids, clf.predict(X_te)))

## 3. Challenge score (validation proxy)

The official score is `50 * macro_f1(task1) + 50 * macro_f1(task2)` on the private test set. We compute the same formula on validation as a proxy.

In [ ]:
proxy = (50 * scores["task1_static"]["macro_f1"]
         + 50 * scores["task2_realtime_prefix5"]["macro_f1"])
print(f"Proxy total score (validation): {proxy:.1f} / 100")

## 4. Write submission.json

The challenge accepts a JSON mapping `flow_id -> app` for each task.

In [ ]:
with open("submission.json", "w") as fh:
    json.dump(submission, fh, indent=2)

counts = {t: len(v) for t, v in submission.items()}
print("Wrote submission.json with predictions:", counts)
dict(list(submission["task1_static"].items())[:3])